# LeetCode 4074 — Maximum Product Pair With Target Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-product-pair-with-target-sum/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Math, Two Pointers

Skeleton LeetCode:

```python
class Solution:
    def maxProductPair(self, nums: List[int], target: int) -> List[int]:
        ...
```

Ghi chú: đề và các ví dụ lấy từ đề gốc (Biweekly Contest 193, câu 1); mọi đáp án trong notebook đều được kiểm chứng bằng code, kể cả bằng một lời giải độc lập duyệt mọi cặp chỉ số có thứ tự. Vì đề cho phép trả **bất kỳ** cặp tối ưu nào, phần test kiểm tra theo **tính hợp lệ và tích lớn nhất**, không chỉ so sánh với một đáp án cố định.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm cặp chỉ số **khác nhau** `(i, j)` sao cho `nums[i] + nums[j] == target` và `nums[i] > nums[j]` (hai giá trị **phải khác nhau**, giá trị lớn đứng ở `i`); trong các cặp hợp lệ chọn cặp có **tích lớn nhất**; không có thì trả `[-1, -1]`.
2. Làm được cách **brute force** duyệt mọi cặp `O(n²)`, và cách **Hash Map** `O(n)` giống Two Sum (nhưng có thêm điều kiện về tích và thứ tự).
3. Nhận ra **bước đột phá toán học**: khi tổng cố định `T`, tích `x·y = (T² − (x − y)²) / 4`, nên **tích lớn nhất ⇔ hiệu `x − y` nhỏ nhất ⇔ hai số gần `T/2` nhất**. Từ đó dùng **sắp xếp + hai con trỏ** hoặc **duyệt từ giá trị gần `T/2`**.
4. Xử lý đúng **số âm**, **giá trị bằng nhau** (không hợp lệ), **trả chỉ số** (không phải giá trị) và **đáp án không duy nhất** (kiểm tra bằng bộ kiểm hợp lệ).
5. Tránh các lỗi kinh điển: cho phép hai giá trị bằng nhau, trả `[j, i]` sai thứ tự, khởi tạo tích lớn nhất bằng `0` (bỏ sót tích âm), và trả giá trị thay vì chỉ số.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` và số nguyên `target`.

Một cặp chỉ số **phân biệt** `(i, j)` là **hợp lệ** nếu:

- `nums[i] + nums[j] == target`;
- `nums[i] > nums[j]`.

Trả về một cặp hợp lệ `[i, j]` có **tích** `nums[i] * nums[j]` **lớn nhất** trong mọi cặp hợp lệ. Nếu không có cặp hợp lệ, trả `[-1, -1]`. Nếu nhiều cặp cùng đạt tích lớn nhất, trả **một cặp bất kỳ** trong số đó.

**Ví dụ 1**

- Input: `nums = [1,2,3,4]`, `target = 5`
- Output: `[2,1]`
- Giải thích: có 2 cặp hợp lệ: `(3, 0)` (`4 + 1 = 5`, tích `4`) và `(2, 1)` (`3 + 2 = 5`, tích `6`). Cặp thứ hai có tích lớn hơn nên đáp án là `[2, 1]`.

**Ví dụ 2**

- Input: `nums = [-3,-1,4,2]`, `target = 1`
- Output: `[3,1]`
- Giải thích: có 2 cặp hợp lệ: `(2, 0)` (`4 + (−3) = 1`, tích `−12`) và `(3, 1)` (`2 + (−1) = 1`, tích `−2`). Vì `−2 > −12` nên đáp án là `[3, 1]`.

**Ví dụ 3**

- Input: `nums = [3,3,5]`, `target = 6`
- Output: `[-1,-1]`
- Giải thích: `nums[0] + nums[1] = 6` nhưng hai giá trị bằng nhau nên không thỏa `nums[i] > nums[j]`. Không có cặp hợp lệ.

**Ràng buộc:** `2 <= nums.length <= 100`; `-100 <= nums[i], target <= 100`.

In [1]:
from typing import List
import random
from math import inf

examples = [
    ([1, 2, 3, 4], 5, [2, 1]),
    ([-3, -1, 4, 2], 1, [3, 1]),
    ([3, 3, 5], 6, [-1, -1]),
]
for nums, target, expected in examples:
    print(f"nums={nums}, target={target}, kỳ vọng {expected}")

nums=[1, 2, 3, 4], target=5, kỳ vọng [2, 1]
nums=[-3, -1, 4, 2], target=1, kỳ vọng [3, 1]
nums=[3, 3, 5], target=6, kỳ vọng [-1, -1]


## 3. Kiến thức nền

### 3.1. Chỉ số và giá trị: đừng nhầm hai thứ

`nums[i]` là **giá trị** ở **chỉ số** `i`. Đề yêu cầu **trả chỉ số** `[i, j]` (không phải giá trị). `enumerate(nums)` cho các cặp `(chỉ số, giá trị)`.

In [2]:
nums = [-3, -1, 4, 2]
for i, x in enumerate(nums):
    print(f"chỉ số {i} -> giá trị {x}")
print("nums[3] =", nums[3], "| nums[1] =", nums[1])

chỉ số 0 -> giá trị -3
chỉ số 1 -> giá trị -1
chỉ số 2 -> giá trị 4
chỉ số 3 -> giá trị 2
nums[3] = 2 | nums[1] = -1


### 3.2. `dict` — tra cứu "giá trị → chỉ số" trong `O(1)` trung bình

Giống Two Sum: với mỗi `x`, ta cần tìm `y = target − x` đã xuất hiện trước đó ở chỉ số nào. `dict` lưu `giá trị → chỉ số`; `y in pos` kiểm tra tồn tại, `pos[y]` lấy chỉ số.

In [3]:
pos = {}
for i, x in enumerate([-3, -1, 4, 2]):
    pos[x] = i                      # nếu giá trị trùng, chỉ số sau ghi đè chỉ số trước
print(pos)                           # {-3: 0, -1: 1, 4: 2, 2: 3}
print(4 in pos, 7 in pos, pos[4])    # True False 2
print({3: 0, 3: 1})                  # khóa trùng: giữ giá trị cuối {3: 1}

{-3: 0, -1: 1, 4: 2, 2: 3}
True False 2
{3: 1}


### 3.3. Tích của hai số âm/dương và so sánh số âm

Tích hai số cùng dấu là dương, khác dấu là âm. So sánh số âm: `-2 > -12` (số âm "nhỏ hơn" khi giá trị tuyệt đối lớn hơn). Vì thế **đừng khởi tạo "tích lớn nhất" bằng `0`**: mọi tích âm sẽ bị bỏ sót. Dùng `None` hoặc `-inf` (`from math import inf`).

In [4]:
print(4 * -3, 2 * -1)                # -12 -2
print(-2 > -12, max(-2, -12))        # True -2
best = -inf
for p in [-12, -2]:
    best = max(best, p)
print("tích lớn nhất:", best)         # -2
print(-inf < -10**9, 0 > -inf)       # True True

-12 -2
True -2
tích lớn nhất: -2
True True


### 3.4. Đồng nhất thức mấu chốt: `x·y = ((x + y)² − (x − y)²) / 4`

Khai triển `(x + y)² − (x − y)² = 4xy`. Nên khi **tổng `x + y = T` cố định**: `x·y = (T² − (x − y)²) / 4`. Phần `T²` không đổi, nên tích **càng lớn khi `(x − y)²` càng nhỏ**, tức hai số **càng gần nhau**, nghĩa là gần `T/2`. Đúng với cả số âm.

In [5]:
T = 5
for x, y in [(4, 1), (3, 2)]:
    print(f"x={x}, y={y}: x*y = {x * y}, (T^2 - (x-y)^2)/4 = {(T * T - (x - y) ** 2) / 4}")
T = 1
for x, y in [(4, -3), (2, -1)]:
    print(f"x={x}, y={y}: x*y = {x * y}, (T^2 - (x-y)^2)/4 = {(T * T - (x - y) ** 2) / 4}, hiệu = {x - y}")

x=4, y=1: x*y = 4, (T^2 - (x-y)^2)/4 = 4.0
x=3, y=2: x*y = 6, (T^2 - (x-y)^2)/4 = 6.0
x=4, y=-3: x*y = -12, (T^2 - (x-y)^2)/4 = -12.0, hiệu = 7
x=2, y=-1: x*y = -2, (T^2 - (x-y)^2)/4 = -2.0, hiệu = 3


### 3.5. Sắp xếp theo giá trị nhưng giữ chỉ số: `sorted(range(n), key=...)`

Để sắp giá trị mà không mất chỉ số gốc, sắp **danh sách chỉ số** theo giá trị: `order = sorted(range(n), key=lambda i: nums[i])`. Khi đó `nums[order[0]]` là giá trị nhỏ nhất và `order[0]` là chỉ số của nó.

In [6]:
nums = [-3, -1, 4, 2]
order = sorted(range(len(nums)), key=lambda i: nums[i])
print(order)                                # [0, 1, 3, 2]
print([nums[i] for i in order])             # [-3, -1, 2, 4]

[0, 1, 3, 2]
[-3, -1, 2, 4]


### 3.6. Hai con trỏ trên mảng đã sắp

Với mảng đã sắp tăng, đặt `l = 0`, `r = n − 1`. Nếu `a[l] + a[r] < target`: tăng `l` (cần tổng lớn hơn). Nếu `> target`: giảm `r`. Nếu bằng: tìm được một cặp; rồi thu hẹp cả hai. Mỗi bước loại bỏ ít nhất một phần tử, nên chạy `O(n)`.

In [7]:
a = [-3, -1, 2, 4]
l, r, target = 0, len(a) - 1, 1
while l < r:
    s = a[l] + a[r]
    print(f"l={l} r={r}: {a[l]} + {a[r]} = {s}", "<" if s < target else (">" if s > target else "== target"))
    if s < target:
        l += 1
    elif s > target:
        r -= 1
    else:
        l += 1
        r -= 1

l=0 r=3: -3 + 4 = 1 == target
l=1 r=2: -1 + 2 = 1 == target


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Ví dụ 2: `nums = [-3, -1, 4, 2]`, `target = 1`.

Liệt kê **mọi** cặp chỉ số `(i, j)` với `i ≠ j`, kiểm tra từng điều kiện:

| `(i, j)` | `nums[i], nums[j]` | tổng `= 1`? | `nums[i] > nums[j]`? | hợp lệ? | tích |
|---|---|---|---|---|---|
| `(0, 1)` | `-3, -1` | `-4` không | | | |
| `(0, 2)` | `-3, 4` | `1` có | `-3 > 4` không | không | |
| `(2, 0)` | `4, -3` | `1` có | `4 > -3` có | có | `-12` |
| `(1, 3)` | `-1, 2` | `1` có | `-1 > 2` không | không | |
| `(3, 1)` | `2, -1` | `1` có | `2 > -1` có | có | `-2` |
| `(2, 3)` | `4, 2` | `6` không | | | |

Có hai cặp hợp lệ, tích `-12` và `-2`; chọn `-2`, tức `[3, 1]`.

Để ý: cả hai cặp hợp lệ đều có tổng 1; cặp được chọn `(2, -1)` có hai số **gần nhau hơn** (hiệu `3`) so với `(4, -3)` (hiệu `7`). Đó là gợi ý cho bước đột phá.

In [8]:
nums, target = [-3, -1, 4, 2], 1
valid = []
for i in range(len(nums)):
    for j in range(len(nums)):
        if i != j and nums[i] + nums[j] == target and nums[i] > nums[j]:
            valid.append((i, j, nums[i] * nums[j], nums[i] - nums[j]))
print("(i, j, tích, hiệu):")
for item in valid:
    print("  ", item)

(i, j, tích, hiệu):
   (2, 0, -12, 7)
   (3, 1, -2, 3)


## 5. Cách 1 — Brute Force / baseline: duyệt mọi cặp chỉ số

**Ý tưởng.** Hai vòng `for` duyệt mọi cặp `(i, j)` với `i ≠ j`. Với cặp thỏa tổng `target` và `nums[i] > nums[j]`, tính tích; nếu lớn hơn tích tốt nhất hiện có thì cập nhật đáp án. Tích tốt nhất khởi tạo bằng `None` (hoặc `-inf`) để tích âm vẫn được xét.

In [9]:
def max_product_pair_bruteforce(nums: list[int], target: int) -> list[int]:
    n = len(nums)
    best = None                                  # chưa có cặp hợp lệ nào
    answer = [-1, -1]
    for i in range(n):
        for j in range(n):
            if i == j:
                continue                          # hai chỉ số phải khác nhau
            if nums[i] + nums[j] == target and nums[i] > nums[j]:
                product = nums[i] * nums[j]
                if best is None or product > best:
                    best = product
                    answer = [i, j]
    return answer


for nums, target, expected in examples:
    print(f"nums={nums}, target={target} -> {max_product_pair_bruteforce(nums, target)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4], target=5 -> [2, 1] (kỳ vọng [2, 1])
nums=[-3, -1, 4, 2], target=1 -> [3, 1] (kỳ vọng [3, 1])
nums=[3, 3, 5], target=6 -> [-1, -1] (kỳ vọng [-1, -1])


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Hai vòng lặp `n × n`, mỗi lần `O(1)`: **Time `O(n²)`**, **Space `O(1)`**. Với `n ≤ 100` chỉ khoảng `10⁴` thao tác.

**Nút thắt.** Với mỗi `i` ta duyệt cả mảng chỉ để tìm `j` thỏa `nums[j] = target − nums[i]`: đây là bài toán "tìm phần bù" của Two Sum, giải được bằng bảng tra cứu. Ngoài ra ta chưa dùng tính chất của **tích** để biết cặp nào tốt nhất mà không cần so sánh mọi cặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Hai ý, giải quyết hai nút thắt.**

**Ý 1 (tìm phần bù nhanh).** Với mỗi `x`, phần bù là `y = target − x`. Dùng `dict` `giá trị → chỉ số` để biết `y` đã xuất hiện ở đâu trong `O(1)`. Duyệt một lượt từ trái sang phải, tại `x` chỉ cần xét các `y` xuất hiện **trước**; mỗi cặp `{x, y}` được xét đúng một lần (khi gặp số xuất hiện sau). Điều kiện hợp lệ: `x ≠ y`. Nếu hợp lệ thì **số lớn đứng ở `i`**, số nhỏ ở `j`.

**Ý 2 (chọn cặp tốt nhất bằng toán).** Từ `x·y = (T² − (x − y)²)/4` (`T = target`): với cùng tổng `T`, **tích lớn hơn ⇔ hiệu `|x − y|` nhỏ hơn**. Vậy cặp tối ưu là cặp hợp lệ có **hiệu nhỏ nhất**: hai giá trị gần `T/2` nhất. Hệ quả:

- sau khi sắp xếp, hai con trỏ từ hai đầu tiến vào: các cặp có tổng `T` được tìm thấy theo thứ tự **hiệu giảm dần**, nên **cặp cuối cùng** tìm thấy là tối ưu (Cách 3);
- hoặc: duyệt giá trị nhỏ `y` từ lớn xuống nhỏ, bắt đầu từ `y < T/2`; `y` đầu tiên có `T − y` tồn tại cho cặp tối ưu (Cách 4).

Hệ quả hay: với mỗi `T`, **cặp giá trị tối ưu là duy nhất** (vì hiệu xác định duy nhất từ tích và `T`); chỉ **chỉ số** có thể khác nhau khi có giá trị trùng lặp, nên đề cho phép trả bất kỳ.

In [10]:
nums, target = [-3, -1, 4, 2], 1
for x, y in [(4, -3), (2, -1)]:
    print(f"cặp giá trị ({x}, {y}): tổng={x + y}, hiệu={x - y}, tích={x * y}")
# cặp tốt nhất là cặp có hiệu nhỏ nhất:
pairs = [(4, -3), (2, -1)]
print("hiệu nhỏ nhất:", min(pairs, key=lambda p: p[0] - p[1]), "| tích lớn nhất:", max(pairs, key=lambda p: p[0] * p[1]))

cặp giá trị (4, -3): tổng=1, hiệu=7, tích=-12
cặp giá trị (2, -1): tổng=1, hiệu=3, tích=-2
hiệu nhỏ nhất: (2, -1) | tích lớn nhất: (2, -1)


## 7. Các cách tiếp theo

### Cách 2 — Hash Map `giá trị → chỉ số gần nhất` (một lượt, cách của lời giải tham khảo)

**Giải thích.**

1. `pos = {}` lưu chỉ số **gần nhất** của mỗi giá trị đã duyệt; `best = None`, `answer = [-1, -1]`.
2. Với mỗi `(i, x)`: `y = target − x`. Nếu `x ≠ y` và `y in pos`: `j = pos[y]`; tích `x·y`; nếu lớn hơn `best` thì cập nhật `best` và `answer = [i, j]` nếu `x > y`, ngược lại `[j, i]` (số lớn đứng trước).
3. Sau khi xét, ghi `pos[x] = i`. (Ghi **sau** khi xét để không ghép `x` với chính nó.)

In [11]:
def max_product_pair_hash(nums: list[int], target: int) -> list[int]:
    pos = {}                                     # giá trị -> chỉ số gần nhất
    best = None
    answer = [-1, -1]
    for i, x in enumerate(nums):
        y = target - x                           # phần bù cần tìm
        if x != y and y in pos:                  # hai giá trị phải khác nhau
            j = pos[y]
            product = x * y
            if best is None or product > best:
                best = product
                answer = [i, j] if x > y else [j, i]     # số lớn hơn đứng ở vị trí đầu
        pos[x] = i                               # lưu sau khi xét
    return answer


for nums, target, expected in examples:
    print(f"nums={nums}, target={target} -> {max_product_pair_hash(nums, target)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4], target=5 -> [2, 1] (kỳ vọng [2, 1])
nums=[-3, -1, 4, 2], target=1 -> [3, 1] (kỳ vọng [3, 1])
nums=[3, 3, 5], target=6 -> [-1, -1] (kỳ vọng [-1, -1])


In [12]:
# Code nộp LeetCode cho Cách 2
from typing import List


class Solution:
    def maxProductPair(self, nums: List[int], target: int) -> List[int]:
        pos = {}
        best = None
        answer = [-1, -1]
        for i, x in enumerate(nums):
            y = target - x
            if x != y and y in pos:
                j = pos[y]
                product = x * y
                if best is None or product > best:
                    best = product
                    answer = [i, j] if x > y else [j, i]
            pos[x] = i
        return answer

**Độ phức tạp Cách 2:** **Time `O(n)`** (một lượt, thao tác `dict` `O(1)` trung bình), **Space `O(n)`** cho `pos`.

### Cách 3 — Sắp xếp + hai con trỏ, dùng bước đột phá "hiệu nhỏ nhất"

**Giải thích.** Sắp chỉ số theo giá trị (`order`). Đặt `l = 0`, `r = n − 1`. Nếu tổng nhỏ hơn `target` thì `l += 1`; lớn hơn thì `r -= 1`; bằng thì: nếu hai giá trị khác nhau, ghi nhớ cặp `[order[r], order[l]]` (giá trị lớn ở `r` đứng trước), rồi thu hẹp cả hai. Cặp ghi nhớ **sau cùng** có hiệu nhỏ nhất, nên tích lớn nhất, không cần so sánh tích.

In [13]:
def max_product_pair_two_pointers(nums: list[int], target: int) -> list[int]:
    n = len(nums)
    order = sorted(range(n), key=lambda i: nums[i])     # chỉ số, sắp theo giá trị
    l, r = 0, n - 1
    answer = [-1, -1]
    while l < r:
        a, b = nums[order[l]], nums[order[r]]            # a <= b
        s = a + b
        if s < target:
            l += 1
        elif s > target:
            r -= 1
        else:
            if a != b:                                   # hai giá trị phải khác nhau
                answer = [order[r], order[l]]            # b (lớn) trước, a (nhỏ) sau
            l += 1                                       # các cặp tiếp theo có hiệu NHỎ hơn
            r -= 1
    return answer


for nums, target, expected in examples:
    print(f"nums={nums}, target={target} -> {max_product_pair_two_pointers(nums, target)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4], target=5 -> [2, 1] (kỳ vọng [2, 1])
nums=[-3, -1, 4, 2], target=1 -> [3, 1] (kỳ vọng [3, 1])
nums=[3, 3, 5], target=6 -> [-1, -1] (kỳ vọng [-1, -1])


**Độ phức tạp Cách 3:** **Time `O(n log n)`** (sắp xếp; hai con trỏ chỉ `O(n)`), **Space `O(n)`** cho `order`.

### Cách 4 — Duyệt giá trị nhỏ `y` từ gần `T/2` trở xuống

**Giải thích.** Gom `pos: giá trị → một chỉ số` (chỉ số bất kỳ). Cặp tối ưu là `(x, y)` với `y < x`, `x + y = T` và `x − y = T − 2y` **nhỏ nhất**, tức `y` **lớn nhất** thỏa `2y < T` và `T − y` có trong mảng. Duyệt `y` giảm dần: giá trị đầu tiên thỏa `2*y < target` và `target − y in pos` cho đáp án.

In [14]:
def max_product_pair_scan_values(nums: list[int], target: int) -> list[int]:
    pos = {x: i for i, x in enumerate(nums)}             # giá trị -> một chỉ số (cái cuối)
    for y in sorted(pos, reverse=True):                  # giá trị nhỏ y, từ lớn xuống nhỏ
        x = target - y
        if 2 * y < target and x in pos:                  # y < x: hai giá trị khác nhau và đúng thứ tự
            return [pos[x], pos[y]]
    return [-1, -1]


for nums, target, expected in examples:
    print(f"nums={nums}, target={target} -> {max_product_pair_scan_values(nums, target)} (kỳ vọng {expected})")

nums=[1, 2, 3, 4], target=5 -> [2, 1] (kỳ vọng [2, 1])
nums=[-3, -1, 4, 2], target=1 -> [3, 1] (kỳ vọng [3, 1])
nums=[3, 3, 5], target=6 -> [-1, -1] (kỳ vọng [-1, -1])


**Độ phức tạp Cách 4:** **Time `O(n log n)`** (sắp các giá trị khác nhau), **Space `O(n)`**.

> Cách 2 là cách nộp bài chuẩn (một lượt, `O(n)`). Cách 3 và Cách 4 cho thấy **toán học** (tối đa hóa tích ⇔ tối thiểu hóa hiệu) cho phép bỏ hẳn việc so sánh tích. Cách 1 là chuẩn đối chiếu.

## 8. TRACE từng bước

Ta theo dõi Cách 2 (Hash Map) trên ví dụ 2 và ví dụ 3; rồi Cách 3 (hai con trỏ) trên ví dụ 1.

In [15]:
def trace_hash(nums, target):
    print(f"nums={nums}, target={target}")
    pos = {}
    best, answer = None, [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        note = ""
        if x == y:
            note = "x == y: không hợp lệ (hai giá trị phải khác nhau)"
        elif y in pos:
            j = pos[y]
            product = x * y
            if best is None or product > best:
                best = product
                answer = [i, j] if x > y else [j, i]
                note = f"ghép với y={y} ở chỉ số {j}: tích {product} > best cũ -> answer = {answer}"
            else:
                note = f"ghép với y={y} ở chỉ số {j}: tích {product} không hơn best = {best}"
        else:
            note = f"chưa thấy y={y}"
        print(f"  i={i}, x={x}, y={y}, pos(trước)={pos}: {note}")
        pos[x] = i
    print("  => đáp án:", answer)
    return answer


assert trace_hash([-3, -1, 4, 2], 1) == [3, 1]
print()
assert trace_hash([3, 3, 5], 6) == [-1, -1]

nums=[-3, -1, 4, 2], target=1
  i=0, x=-3, y=4, pos(trước)={}: chưa thấy y=4
  i=1, x=-1, y=2, pos(trước)={-3: 0}: chưa thấy y=2
  i=2, x=4, y=-3, pos(trước)={-3: 0, -1: 1}: ghép với y=-3 ở chỉ số 0: tích -12 > best cũ -> answer = [2, 0]
  i=3, x=2, y=-1, pos(trước)={-3: 0, -1: 1, 4: 2}: ghép với y=-1 ở chỉ số 1: tích -2 > best cũ -> answer = [3, 1]
  => đáp án: [3, 1]

nums=[3, 3, 5], target=6
  i=0, x=3, y=3, pos(trước)={}: x == y: không hợp lệ (hai giá trị phải khác nhau)
  i=1, x=3, y=3, pos(trước)={3: 0}: x == y: không hợp lệ (hai giá trị phải khác nhau)
  i=2, x=5, y=1, pos(trước)={3: 1}: chưa thấy y=1
  => đáp án: [-1, -1]


In [16]:
def trace_two_pointers(nums, target):
    n = len(nums)
    order = sorted(range(n), key=lambda i: nums[i])
    print(f"nums={nums}, target={target}, order={order} -> giá trị đã sắp {[nums[i] for i in order]}")
    l, r = 0, n - 1
    answer = [-1, -1]
    while l < r:
        a, b = nums[order[l]], nums[order[r]]
        s = a + b
        if s < target:
            print(f"  l={l} r={r}: {a} + {b} = {s} < {target} -> l += 1")
            l += 1
        elif s > target:
            print(f"  l={l} r={r}: {a} + {b} = {s} > {target} -> r -= 1")
            r -= 1
        else:
            if a != b:
                answer = [order[r], order[l]]
                print(f"  l={l} r={r}: {a} + {b} = {s} == {target}, hiệu {b - a}, tích {a * b} -> ghi nhớ {answer}")
            else:
                print(f"  l={l} r={r}: {a} + {b} == {target} nhưng bằng nhau -> bỏ")
            l += 1
            r -= 1
    print("  => đáp án:", answer)
    return answer


assert trace_two_pointers([1, 2, 3, 4], 5) == [2, 1]

nums=[1, 2, 3, 4], target=5, order=[0, 1, 2, 3] -> giá trị đã sắp [1, 2, 3, 4]
  l=0 r=3: 1 + 4 = 5 == 5, hiệu 3, tích 4 -> ghi nhớ [3, 0]
  l=1 r=2: 2 + 3 = 5 == 5, hiệu 1, tích 6 -> ghi nhớ [2, 1]
  => đáp án: [2, 1]


Bảng tóm tắt (Cách 2, ví dụ 2: `nums = [-3, -1, 4, 2]`, `target = 1`):

| `i` | `x` | `y = 1 − x` | `y in pos`? | hành động | `best` | `answer` | `pos` sau |
|---|---|---|---|---|---|---|---|
| 0 | `-3` | `4` | không | chỉ lưu | `None` | `[-1,-1]` | `{-3:0}` |
| 1 | `-1` | `2` | không | chỉ lưu | `None` | `[-1,-1]` | `{-3:0,-1:1}` |
| 2 | `4` | `-3` | có (`j=0`) | tích `-12`, `4 > -3` nên `[2,0]` | `-12` | `[2,0]` | `{...,4:2}` |
| 3 | `2` | `-1` | có (`j=1`) | tích `-2 > -12`, `2 > -1` nên `[3,1]` | `-2` | `[3,1]` | `{...,2:3}` |

Chú ý ở `i = 2`, tích `-12` **vẫn được nhận** vì `best` ban đầu là `None` (nếu khởi tạo `best = 0` thì `-12` bị bỏ).

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (Hash Map một lượt). Đúng class `Solution`, đúng method `maxProductPair(self, nums: List[int], target: int) -> List[int]`.

In [17]:
from typing import List


class Solution:
    def maxProductPair(self, nums: List[int], target: int) -> List[int]:
        pos = {}                                  # giá trị -> chỉ số gần nhất đã duyệt
        best = None                               # tích lớn nhất đã thấy (None: chưa có cặp hợp lệ)
        answer = [-1, -1]
        for i, x in enumerate(nums):
            y = target - x                        # phần bù
            if x != y and y in pos:               # hai giá trị phải khác nhau (nums[i] > nums[j])
                j = pos[y]
                product = x * y
                if best is None or product > best:
                    best = product
                    answer = [i, j] if x > y else [j, i]    # số lớn hơn đứng ở vị trí đầu
            pos[x] = i                            # lưu sau khi xét
        return answer


for nums, target, expected in examples:
    assert Solution().maxProductPair(nums, target) == expected
print("PASS ví dụ đề cho class Solution")

PASS ví dụ đề cho class Solution


## 10. Vì sao thuật toán đúng

- **Mọi cặp hợp lệ được xét:** một cặp hợp lệ `(i, j)` có `nums[i] + nums[j] = target`, `nums[i] > nums[j]`, nên hai giá trị khác nhau. Gọi `p < q` là chỉ số nhỏ hơn và lớn hơn của hai phần tử. Khi duyệt tới `q`, `y = target − nums[q] = nums[p]` đã nằm trong `pos`; `pos[y]` là chỉ số **gần nhất** của giá trị `y` trước `q`, có thể không phải `p` nhưng cho cùng giá trị `y`, tức cùng tích và cùng tổng, nên là cặp hợp lệ tương đương. Vậy mọi **cặp giá trị** hợp lệ được xét.
- **Điều kiện `x ≠ y`:** `nums[i] > nums[j]` buộc hai giá trị khác nhau; nếu `x == y` thì không hợp lệ dù tổng đúng.
- **Thứ tự `[i, j]`:** `answer = [i, j] if x > y else [j, i]` đảm bảo giá trị ở chỉ số đầu lớn hơn.
- **Chọn tích lớn nhất:** ta so sánh với `best`, bắt đầu `None`, nên tích âm (khi mọi cặp có một số âm) vẫn được nhận. Nhiều cặp cùng tích: ta giữ cặp gặp đầu tiên (`>` chứ không phải `>=`); đề cho phép bất kỳ.
- **Cách 3 và 4 đúng nhờ đồng nhất thức** `x·y = (T² − (x − y)²)/4` với `x + y = T`: tích tăng khi `x − y` (dương) giảm, kể cả với số âm. Hai con trỏ liệt kê các cặp tổng `T` theo hiệu giảm dần (`a` tăng, `b` giảm), nên cặp cuối cùng có hiệu nhỏ nhất; Cách 4 chọn `y` lớn nhất dưới `T/2`, cũng hiệu nhỏ nhất.
- **Không có cặp hợp lệ:** `answer` giữ `[-1, -1]`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`.

- **Cách 1 (mọi cặp):** Time `O(n²)`, Space `O(1)`.
- **Cách 2 (Hash Map một lượt):** Time `O(n)` trung bình (`dict` truy cập `O(1)` trung bình; trường hợp xấu về lý thuyết `O(n²)` khi băm va chạm nặng nhưng không xảy ra thực tế với số nguyên nhỏ), Space `O(n)`.
- **Cách 3 (sắp xếp + hai con trỏ):** Time `O(n log n)`, Space `O(n)`.
- **Cách 4 (duyệt giá trị nhỏ từ gần `T/2`):** Time `O(n log n)`, Space `O(n)`.
- Do `n ≤ 100`, mọi cách đều rất nhanh; các phân tích trên quan trọng khi mở rộng `n` lớn.

## 12. Bộ test

Vì đề cho phép **nhiều đáp án**, ta không chỉ so sánh với một mảng cố định mà viết **bộ kiểm `check_answer`**: (1) nếu không có cặp hợp lệ thì phải ra `[-1, -1]`; (2) nếu có thì cặp trả về phải **hợp lệ** (chỉ số khác nhau, tổng đúng, giá trị đầu lớn hơn) và có **tích bằng tích lớn nhất** tính độc lập bằng `oracle`. `oracle` duyệt mọi cặp chỉ số **có thứ tự** và không dùng `dict` hay sắp xếp.

In [18]:
sol = Solution()
impls = {
    "bruteforce": max_product_pair_bruteforce,
    "hash": max_product_pair_hash,
    "two_pointers": max_product_pair_two_pointers,
    "scan_values": max_product_pair_scan_values,
    "submit": sol.maxProductPair,
}


def oracle(nums, target):
    """Tích lớn nhất trong các cặp hợp lệ, hoặc None nếu không có (duyệt mọi cặp chỉ số có thứ tự)."""
    best = None
    n = len(nums)
    for i in range(n):
        for j in range(n):
            if i != j and nums[i] + nums[j] == target and nums[i] > nums[j]:
                p = nums[i] * nums[j]
                best = p if best is None else max(best, p)
    return best


def check_answer(nums, target, ans):
    best = oracle(nums, target)
    if best is None:
        return ans == [-1, -1]
    if not (isinstance(ans, list) and len(ans) == 2):
        return False
    i, j = ans
    n = len(nums)
    return (0 <= i < n and 0 <= j < n and i != j
            and nums[i] + nums[j] == target and nums[i] > nums[j]
            and nums[i] * nums[j] == best)


for nums, target, expected in examples:
    for name, fn in impls.items():
        got = fn(list(nums), target)
        assert got == expected, (name, nums, target, got, expected)    # ví dụ đề có đáp án duy nhất
        assert check_answer(nums, target, got)
print("PASS ví dụ của đề (mọi cách; trùng đáp án đề và qua bộ kiểm hợp lệ)")

PASS ví dụ của đề (mọi cách; trùng đáp án đề và qua bộ kiểm hợp lệ)


### 12.1. Các trường hợp biên

In [19]:
def check_all(nums, target, label):
    for name, fn in impls.items():
        original = list(nums)
        got = fn(original, target)
        assert check_answer(nums, target, got), (label, name, nums, target, got)
        assert original == nums, (label, name, "đã sửa đầu vào")
    best = oracle(nums, target)
    print("PASS", label, "-> tích lớn nhất:", best)


check_all([1, 2], 3, "n = 2, một cặp hợp lệ")
check_all([2, 1], 3, "n = 2, thứ tự đảo (giá trị lớn đứng trước)")
check_all([2, 2], 4, "n = 2, hai giá trị bằng nhau: không hợp lệ")
check_all([1, 4], 3, "n = 2, không có cặp tổng đúng")
check_all([0, 5], 5, "có số 0: tích 0")
check_all([-5, 5], 0, "target = 0: cặp đối nhau, tích âm")
check_all([-100, 100], 0, "biên -100 và 100")
check_all([-100, -100, -99], -199, "hai số âm: tổng -199, tích 9900")
check_all([1, 4, 4, 1], 5, "giá trị trùng lặp: nhiều cặp chỉ số cùng tích (chấp nhận bất kỳ)")
check_all([5, 1, 2, 3, 4], 6, "nhiều cặp: (5,1) (4,2) -> chọn (4,2), tích 8")
check_all([-3, -1, 4, 2], 1, "tích âm (ví dụ 2 của đề)")
check_all([3, 3, 3, 3], 6, "mọi giá trị bằng nhau")
check_all(list(range(-50, 50)), 1, "n = 100 mọi giá trị khác nhau, target = 1")

PASS n = 2, một cặp hợp lệ -> tích lớn nhất: 2
PASS n = 2, thứ tự đảo (giá trị lớn đứng trước) -> tích lớn nhất: 2
PASS n = 2, hai giá trị bằng nhau: không hợp lệ -> tích lớn nhất: None
PASS n = 2, không có cặp tổng đúng -> tích lớn nhất: None
PASS có số 0: tích 0 -> tích lớn nhất: 0
PASS target = 0: cặp đối nhau, tích âm -> tích lớn nhất: -25
PASS biên -100 và 100 -> tích lớn nhất: -10000
PASS hai số âm: tổng -199, tích 9900 -> tích lớn nhất: 9900
PASS giá trị trùng lặp: nhiều cặp chỉ số cùng tích (chấp nhận bất kỳ) -> tích lớn nhất: 4
PASS nhiều cặp: (5,1) (4,2) -> chọn (4,2), tích 8 -> tích lớn nhất: 8
PASS tích âm (ví dụ 2 của đề) -> tích lớn nhất: -2
PASS mọi giá trị bằng nhau -> tích lớn nhất: None
PASS n = 100 mọi giá trị khác nhau, target = 1 -> tích lớn nhất: 0


### 12.2. Tính chất và phản ví dụ

In [20]:
random.seed(4074)

# Tính chất 1 (hệ quả của x*y = (T^2 - (x-y)^2)/4): cặp GIÁ TRỊ tối ưu là DUY NHẤT.
# Vậy mọi cách phải cho cùng (nums[i], nums[j]) dù chỉ số có thể khác.
for _ in range(500):
    n = random.randint(2, 14)
    nums = [random.randint(-6, 6) for _ in range(n)]
    target = random.randint(-8, 8)
    results = {name: fn(list(nums), target) for name, fn in impls.items()}
    value_pairs = {tuple(nums[i] for i in ans) if ans != [-1, -1] else None for ans in results.values()}
    assert len(value_pairs) == 1, (nums, target, results)
print("PASS tính chất: mọi cách cho cùng cặp giá trị (chỉ số có thể khác khi có giá trị trùng)")

# Tính chất 2: cặp tối ưu có hiệu nhỏ nhất trong các cặp hợp lệ
for _ in range(500):
    n = random.randint(2, 14)
    nums = [random.randint(-6, 6) for _ in range(n)]
    target = random.randint(-8, 8)
    ans = sol.maxProductPair(nums, target)
    if ans == [-1, -1]:
        continue
    diffs = [nums[i] - nums[j] for i in range(n) for j in range(n)
             if i != j and nums[i] + nums[j] == target and nums[i] > nums[j]]
    assert nums[ans[0]] - nums[ans[1]] == min(diffs)
print("PASS tính chất: cặp tối ưu là cặp có hiệu nhỏ nhất")

# Phản ví dụ: cho phép hai giá trị bằng nhau
def sai_cho_phep_bang_nhau(nums, target):
    pos, best, answer = {}, None, [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        if y in pos:                       # thiếu điều kiện x != y
            j = pos[y]
            if best is None or x * y > best:
                best, answer = x * y, [i, j] if x >= y else [j, i]
        pos[x] = i
    return answer

print("[3,3,5], 6 -> sai:", sai_cho_phep_bang_nhau([3, 3, 5], 6), "| đúng:", sol.maxProductPair([3, 3, 5], 6))
assert sai_cho_phep_bang_nhau([3, 3, 5], 6) != [-1, -1]

PASS tính chất: mọi cách cho cùng cặp giá trị (chỉ số có thể khác khi có giá trị trùng)
PASS tính chất: cặp tối ưu là cặp có hiệu nhỏ nhất
[3,3,5], 6 -> sai: [1, 0] | đúng: [-1, -1]


### 12.3. Đối chiếu ngẫu nhiên (có seed) với oracle độc lập

In [21]:
random.seed(2026)
for trial in range(3000):
    n = random.randint(2, 12)
    lo, hi = random.choice([(-3, 3), (-10, 10), (-100, 100), (0, 5)])
    nums = [random.randint(lo, hi) for _ in range(n)]
    target = random.randint(max(-100, 2 * lo), min(100, 2 * hi))
    for name, fn in impls.items():
        got = fn(list(nums), target)
        assert check_answer(nums, target, got), (name, nums, target, got)

for trial in range(300):                           # n tới 100
    n = random.randint(50, 100)
    nums = [random.randint(-100, 100) for _ in range(n)]
    target = random.randint(-100, 100)
    for name, fn in impls.items():
        got = fn(list(nums), target)
        assert check_answer(nums, target, got), (name, target, got)
print("PASS 3300 input ngẫu nhiên: mọi cách hợp lệ và tích lớn nhất khớp oracle độc lập")

PASS 3300 input ngẫu nhiên: mọi cách hợp lệ và tích lớn nhất khớp oracle độc lập


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Cho phép hai giá trị bằng nhau

In [22]:
def sai_bang_nhau(nums, target):
    pos, best, answer = {}, None, [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        if y in pos:                                  # quên điều kiện x != y
            j = pos[y]
            if best is None or x * y > best:
                best = x * y
                answer = [i, j] if x >= y else [j, i]
        pos[x] = i
    return answer


print("sai:", sai_bang_nhau([3, 3, 5], 6), "| đúng:", Solution().maxProductPair([3, 3, 5], 6))

sai: [1, 0] | đúng: [-1, -1]


**Vì sao sai:** đề yêu cầu `nums[i] > nums[j]` **chặt**. Hai chỉ số khác nhau nhưng cùng giá trị `3` (tổng `3 + 3 = 6`) không thỏa. Ví dụ 3 phải trả `[-1, -1]`. Điều kiện `x != y` loại trường hợp này.

### Lỗi 2: Trả `[j, i]` thay vì số lớn đứng trước

In [23]:
def sai_thu_tu(nums, target):
    pos, best, answer = {}, None, [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        if x != y and y in pos:
            j = pos[y]
            if best is None or x * y > best:
                best = x * y
                answer = [j, i]                       # sai: luôn [chỉ số cũ, chỉ số mới]
        pos[x] = i
    return answer


print("[1,2,3,4], 5 -> sai:", sai_thu_tu([1, 2, 3, 4], 5), "| đúng:", Solution().maxProductPair([1, 2, 3, 4], 5))

[1,2,3,4], 5 -> sai: [1, 2] | đúng: [2, 1]


**Vì sao sai:** cặp trả về `[i, j]` cần `nums[i] > nums[j]`. Chỉ số xuất hiện **trước** (`j` trong code) có thể chứa giá trị nhỏ hoặc lớn tùy dữ liệu. Với `[1,2,3,4]`: `x = 3` (chỉ số 2) ghép với `y = 2` (chỉ số 1): `x > y` nên phải trả `[2, 1]`, không phải `[1, 2]`. Phải so sánh `x` và `y` rồi mới sắp thứ tự.

### Lỗi 3: Khởi tạo tích lớn nhất bằng `0`

In [24]:
def sai_best_bang_0(nums, target):
    pos, best, answer = {}, 0, [-1, -1]               # sai: tích âm không bao giờ > 0
    for i, x in enumerate(nums):
        y = target - x
        if x != y and y in pos:
            j = pos[y]
            if x * y > best:
                best = x * y
                answer = [i, j] if x > y else [j, i]
        pos[x] = i
    return answer


print("[-3,-1,4,2], 1 -> sai:", sai_best_bang_0([-3, -1, 4, 2], 1), "| đúng:", Solution().maxProductPair([-3, -1, 4, 2], 1))

[-3,-1,4,2], 1 -> sai: [-1, -1] | đúng: [3, 1]


**Vì sao sai:** khi mọi cặp hợp lệ có tích **âm** (như ví dụ 2: `-12` và `-2`), điều kiện `tích > 0` không bao giờ đúng nên trả `[-1, -1]`. Khởi tạo `best = None` (hoặc `-inf`) và so sánh `best is None or tích > best`. Đây là lỗi số âm rất hay gặp.

### Lỗi 4: Trả về giá trị thay vì chỉ số

In [25]:
def sai_tra_gia_tri(nums, target):
    ans = Solution().maxProductPair(nums, target)
    if ans == [-1, -1]:
        return ans
    return [nums[ans[0]], nums[ans[1]]]               # sai: đề cần chỉ số


print("[1,2,3,4], 5 -> sai:", sai_tra_gia_tri([1, 2, 3, 4], 5), "| đúng:", Solution().maxProductPair([1, 2, 3, 4], 5))

[1,2,3,4], 5 -> sai: [3, 2] | đúng: [2, 1]


**Vì sao sai:** đề yêu cầu **chỉ số** `[i, j]`. Trả giá trị `[3, 2]` thay vì chỉ số `[2, 1]` sẽ bị chấm sai (và còn có thể trùng hợp đúng khi `nums` là hoán vị bắt đầu từ 0, che giấu lỗi). Hãy luôn tự hỏi "đề cần chỉ số hay giá trị?" khi đọc đề.

## 14. Mẹo làm nhanh & nhận dạng pattern

- Thấy "**hai phần tử có tổng bằng `target`**": nghĩ ngay **Two Sum**: `y = target − x` và `dict` hoặc **sắp xếp + hai con trỏ**.
- "Tối đa hóa **tích** của hai số có **tổng cố định**" → hai số **gần nhau nhất** (`x·y = (T² − (x−y)²)/4`). Ngược lại, **tối thiểu hóa tích** là hai số **xa nhau nhất**.
- Khi có **số âm**, đừng khởi tạo "lớn nhất" bằng `0`; dùng `None` hoặc `-inf`.
- Đề cho phép "trả bất kỳ": viết **bộ kiểm hợp lệ** (`check_answer`) thay vì so sánh trực tiếp với một mảng.
- `nums[i] > nums[j]` chặt nghĩa là loại cặp bằng nhau: thêm `x != y`.
- Lưu `dict` **sau khi** xét `x` để không ghép `x` với chính nó (chỉ số khác nhau).
- Nhớ bài tập đối xứng: tổng cố định + tối thiểu hóa tích, hoặc tích cố định + tối thiểu hóa tổng (AM-GM).

## 15. Biến thể của bài

1. **Đếm số cặp hợp lệ** (tổng `target`, `nums[i] > nums[j]`).
2. **Tích nhỏ nhất** thay vì lớn nhất (hai số xa nhau nhất).
3. **Liệt kê mọi cặp chỉ số** đạt tích lớn nhất.
4. **Cho phép hai giá trị bằng nhau** (bỏ điều kiện `>`; chỉ số khác nhau).
5. **Two Sum cổ điển** (LeetCode 1): trả hai chỉ số có tổng `target`.

In [26]:
def count_valid_pairs(nums, target):
    """Biến thể 1: số cặp chỉ số (i, j) hợp lệ."""
    from collections import Counter
    cnt = Counter(nums)
    total = 0
    for y in cnt:
        x = target - y
        if y < x and x in cnt:
            total += cnt[x] * cnt[y]
    return total


def min_product_pair(nums, target):
    """Biến thể 2: tích nhỏ nhất, đáp án dạng [i, j] (bất kỳ)."""
    best, answer = None, [-1, -1]
    n = len(nums)
    for i in range(n):
        for j in range(n):
            if i != j and nums[i] + nums[j] == target and nums[i] > nums[j]:
                p = nums[i] * nums[j]
                if best is None or p < best:
                    best, answer = p, [i, j]
    return answer


def all_best_pairs(nums, target):
    """Biến thể 3: mọi cặp chỉ số có tích lớn nhất."""
    best = oracle(nums, target)
    n = len(nums)
    return [[i, j] for i in range(n) for j in range(n)
            if i != j and nums[i] + nums[j] == target and nums[i] > nums[j] and nums[i] * nums[j] == best]


def max_product_allow_equal(nums, target):
    """Biến thể 4: cho phép hai giá trị bằng nhau (chỉ số khác nhau)."""
    pos, best, answer = {}, None, [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        if y in pos:
            j = pos[y]
            if best is None or x * y > best:
                best, answer = x * y, [j, i]
        pos[x] = i
    return answer


def two_sum(nums, target):
    """Biến thể 5: Two Sum (LeetCode 1)."""
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
    return []


assert count_valid_pairs([1, 2, 3, 4], 5) == 2
assert count_valid_pairs([3, 3, 5], 6) == 0
assert count_valid_pairs([1, 4, 4, 1], 5) == 4
assert min_product_pair([1, 2, 3, 4], 5) == [3, 0]
assert sorted(all_best_pairs([1, 4, 4, 1], 5)) == [[1, 0], [1, 3], [2, 0], [2, 3]]
assert max_product_allow_equal([3, 3, 5], 6) == [0, 1]
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
print("PASS biến thể")

PASS biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi cặp chỉ số | hai vòng `for`, so sánh tích | `O(n²)` | `O(1)` | hiểu đề; oracle đối chiếu |
| 2. Hash Map một lượt | `y = target − x`, `pos[y]`, cập nhật tích lớn nhất | `O(n)` | `O(n)` | **cách nộp bài** |
| 3. Sắp xếp + hai con trỏ | cặp tìm thấy cuối cùng có hiệu nhỏ nhất | `O(n log n)` | `O(n)` | thấy được cấu trúc toán học |
| 4. Duyệt `y` từ gần `T/2` | `y` lớn nhất dưới `T/2` mà `T − y` tồn tại | `O(n log n)` | `O(n)` | ngắn gọn, ý "hiệu nhỏ nhất" |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `4 * -3`, `max(-2, -12)`, `Solution().maxProductPair([2, 1], 3)`, `Solution().maxProductPair([2, 2], 4)`, `Solution().maxProductPair([1, 4], 3)`.

**Bài 2 (tính tay).** Với `nums = [5, 1, 2, 3, 4]`, `target = 6`: liệt kê các cặp giá trị hợp lệ, hiệu và tích của chúng, rồi cho biết đáp án chỉ số. Làm lại với `nums = [-2, -4, 1, 3]`, `target = -1`.

**Bài 3 (điền TODO).** Hoàn thành `pair_todo(nums, target)` theo Cách 2.

In [27]:
def pair_todo(nums, target):
    pos = {}
    best = None
    answer = [-1, -1]
    for i, x in enumerate(nums):
        # TODO 1: y = target - x
        # TODO 2: nếu x != y và y in pos thì lấy j = pos[y]
        # TODO 3: tính tích x * y; nếu best is None hoặc tích > best thì cập nhật best và answer (số lớn trước)
        # TODO 4: pos[x] = i (lưu SAU khi xét)
        pass
    return answer

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `pair_two_pointers2(nums, target)` theo Cách 3 (sắp chỉ số theo giá trị rồi hai con trỏ) mà không nhìn code mẫu. Kiểm tra bằng bộ kiểm hợp lệ trên ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_pairs(nums, target)` đếm số cặp chỉ số `(i, j)` hợp lệ. (b) Viết `min_product(nums, target)` trả về **giá trị** tích nhỏ nhất của các cặp hợp lệ (hoặc `None`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: số âm nhân số dương cho số âm; `max` chọn số lớn hơn; cặp phải có `nums[i] > nums[j]`; hai giá trị bằng nhau không hợp lệ. Bài 2: tích lớn nhất khi hai số gần nhau nhất.
- **Mức 2 (vừa):** Bài 1: `-12`, `-2`, `[0, 1]`, `[-1, -1]`, `[-1, -1]`. Bài 2: `[5,1,2,3,4]`, `T = 6`: cặp giá trị `(5,1)` hiệu 4 tích 5; `(4,2)` hiệu 2 tích 8; nên chọn `(4, 2)` ứng với chỉ số `[4, 2]`. `[-2,-4,1,3]`, `T = -1`: cặp `(3,-4)` hiệu 7 tích -12; `(1,-2)` hiệu 3 tích -2; chọn `(1, -2)` ứng với `[2, 0]`. Bài 3: bốn TODO trong comment.
- **Mức 3 (sâu):** Bài 4: `order = sorted(range(n), key=lambda i: nums[i])`; `answer = [order[r], order[l]]` khi bằng và hai giá trị khác nhau, rồi `l += 1`, `r -= 1`. Bài 5(a): đếm bằng hai vòng lặp hoặc `Counter` với `y < target − y`. Bài 5(b): giống brute force nhưng lấy `min`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 4 * -3 == -12
assert max(-2, -12) == -2
assert Solution().maxProductPair([2, 1], 3) == [0, 1]
assert Solution().maxProductPair([2, 2], 4) == [-1, -1]
assert Solution().maxProductPair([1, 4], 3) == [-1, -1]

# Bài 2
assert Solution().maxProductPair([5, 1, 2, 3, 4], 6) == [4, 2]
assert check_answer([5, 1, 2, 3, 4], 6, [4, 2])
assert Solution().maxProductPair([-2, -4, 1, 3], -1) == [2, 0]
assert oracle([-2, -4, 1, 3], -1) == -2

# Bài 3
def pair_todo_ans(nums, target):
    pos = {}
    best = None
    answer = [-1, -1]
    for i, x in enumerate(nums):
        y = target - x
        if x != y and y in pos:
            j = pos[y]
            product = x * y
            if best is None or product > best:
                best = product
                answer = [i, j] if x > y else [j, i]
        pos[x] = i
    return answer

for nums, target, expected in examples:
    assert pair_todo_ans(nums, target) == expected

# Bài 4
def pair_two_pointers2(nums, target):
    n = len(nums)
    order = sorted(range(n), key=lambda i: nums[i])
    l, r = 0, n - 1
    answer = [-1, -1]
    while l < r:
        a, b = nums[order[l]], nums[order[r]]
        if a + b < target:
            l += 1
        elif a + b > target:
            r -= 1
        else:
            if a != b:
                answer = [order[r], order[l]]
            l += 1
            r -= 1
    return answer

for nums, target, expected in examples:
    assert check_answer(nums, target, pair_two_pointers2(nums, target))
random.seed(19)
for _ in range(300):
    nums = [random.randint(-8, 8) for _ in range(random.randint(2, 12))]
    target = random.randint(-10, 10)
    assert check_answer(nums, target, pair_two_pointers2(nums, target))

# Bài 5(a), 5(b)
def count_pairs(nums, target):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(n)
               if i != j and nums[i] + nums[j] == target and nums[i] > nums[j])

def min_product(nums, target):
    n = len(nums)
    products = [nums[i] * nums[j] for i in range(n) for j in range(n)
                if i != j and nums[i] + nums[j] == target and nums[i] > nums[j]]
    return min(products) if products else None

assert count_pairs([1, 2, 3, 4], 5) == 2 and count_pairs([3, 3, 5], 6) == 0
assert min_product([1, 2, 3, 4], 5) == 4 and min_product([-3, -1, 4, 2], 1) == -12
assert min_product([3, 3, 5], 6) is None
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (Two Sum một lượt + chọn tích lớn nhất, hai giá trị khác nhau)

```python
class Solution:
    def maxProductPair(self, nums: List[int], target: int) -> List[int]:
        pos = {}
        best = None
        answer = [-1, -1]
        for i, x in enumerate(nums):
            y = target - x
            if x != y and y in pos:
                j = pos[y]
                if best is None or x * y > best:
                    best = x * y
                    answer = [i, j] if x > y else [j, i]
            pos[x] = i
        return answer
```

**Checklist**

- [ ] Hai **giá trị** phải khác nhau (`x != y`), vì `nums[i] > nums[j]` chặt?
- [ ] Số **lớn đứng trước** trong `[i, j]` (so sánh `x` và `y`)?
- [ ] `best` khởi tạo `None`/`-inf` (không phải `0`) để xử lý tích âm?
- [ ] Trả **chỉ số**, không trả giá trị?
- [ ] Lưu `pos[x] = i` **sau khi** xét?
- [ ] Đã thử: hai giá trị bằng nhau, tích âm, `target = 0`, giá trị trùng lặp, `n = 2`, không có cặp?
- [ ] Đã đối chiếu bằng bộ kiểm hợp lệ + brute force (vì đáp án không duy nhất)?
- [ ] Skeleton đúng `class Solution` / `def maxProductPair(self, nums: List[int], target: int) -> List[int]`?